In [ ]:
SENSORS = [
    {"name": "bearing1", "port": "COM3", "baud": 115200},
    {"name": "bearing2", "port": "COM4", "baud": 115200},
    {"name": "bearing3", "port": "COM5", "baud": 115200},
]

In [ ]:
"""
collect_features.py  —  PIPELINE MULTI-CAPTEURS EN SÉRIE
--------------------------------------------------------
Surveillance d'anomalie sur PLUSIEURS capteurs de roulement, analysés les uns
après les autres (en série).

Déroulement complet :
  1. RODAGE GLOBAL : on attend WARMUP_DURATION (20 min) UNE SEULE FOIS,
     le temps que la machine se stabilise.
  2. Pour CHAQUE capteur, en série :
        a) COLLECTE   (COLLECT_DURATION = 30 min) -> features_<nom>.npy
        b) ENTRAÎNEMENT de l'autoencodeur          -> model_<nom>.keras
                                                      threshold_<nom>.json
  3. DÉTECTION DE PROBLÈME, en série : on surveille chaque capteur
     (DETECT_DURATION) et on rend un verdict (sain / problème détecté).

Autoencodeur : 14 -> 8 -> 4 -> 8 -> 14 (ReLU, sortie linéaire), perte MSE.
Normalisation : RobustScaler. Seuil d'anomalie = mean + K_SIGMA*std de l'erreur
de reconstruction sur les données d'entraînement.

Dépendances :  pip install pyserial scipy numpy scikit-learn tensorflow

Utilisation (Jupyter ou script) :
    run_pipeline(SENSORS)                       # capteurs réels
    run_pipeline(SENSORS_SIM, pause_between=False)   # test sans matériel
"""

import time
import json
import threading
import queue
from pathlib import Path

import numpy as np
import serial                              # pip install pyserial
from scipy.stats import kurtosis, skew     # pip install scipy
# NB : tensorflow et scikit-learn sont importés à la demande (dans train/detect)
#      pour que la phase de collecte reste utilisable même sans eux.

# ─── PARAMÈTRES ────────────────────────────────────────────────────────────────

WARMUP_DURATION  = 20 * 60        # RODAGE GLOBAL : 20 min, une seule fois
COLLECT_DURATION = 30 * 60        # COLLECTE : 30 min PAR capteur
DETECT_DURATION  = 60             # DÉTECTION : 60 s de surveillance PAR capteur

SAMPLING_RATE    = 10_000         # Hz — à adapter à ton capteur
WINDOW_SIZE      = 1024           # samples par fenêtre (≈0.1s à 10kHz)
WINDOW_OVERLAP   = 512            # chevauchement 50%
WINDOW_STEP      = WINDOW_SIZE - WINDOW_OVERLAP

K_SIGMA          = 3.0            # seuil = mean + K_SIGMA*std de l'erreur
ANOMALY_FRACTION = 0.10           # > 10% de fenêtres au-dessus du seuil => problème
EPOCHS           = 50
BATCH_SIZE       = 64

OUTPUT_DIR       = Path(".")

# ─── LISTE DES CAPTEURS (analysés en série, dans cet ordre) ─────────────────────
# Un capteur = un nom + un port COM. baud optionnel (115200 par défaut).
SENSORS = [
    {"name": "bearing1", "port": "COM3", "baud": 115200},
    {"name": "bearing2", "port": "COM4", "baud": 115200},
    {"name": "bearing3", "port": "COM5", "baud": 115200},
    # ... ajoute autant de capteurs que nécessaire
]

# Version simulée (pour tester sans matériel)
SENSORS_SIM = [
    {"name": "bearing1", "simulate": True},
    {"name": "bearing2", "simulate": True},
    {"name": "bearing3", "simulate": True},
]

# ─── EXTRACTION DES 14 FEATURES ────────────────────────────────────────────────

FEATURE_NAMES = [
    "mean", "std", "rms", "max", "min",
    "peak_to_peak", "skewness", "kurtosis",
    "crest_factor", "shape_factor", "impulse_factor",
    "freq_dominant", "spectral_energy", "spectral_entropy",
]


def extract_features(window: np.ndarray, fs: int = SAMPLING_RATE) -> np.ndarray:
    """Calcule les 14 features sur une fenêtre temporelle. Retourne shape (14,)."""
    n = len(window)
    eps = 1e-10

    mean_val    = np.mean(window)
    std_val     = np.std(window)
    rms_val     = np.sqrt(np.mean(window ** 2))
    max_val     = np.max(np.abs(window))
    min_val     = np.min(np.abs(window))
    p2p_val     = np.ptp(window)
    skew_val    = float(skew(window))
    kurt_val    = float(kurtosis(window))
    crest_val   = max_val / (rms_val + eps)
    mean_abs    = np.mean(np.abs(window))
    shape_val   = rms_val  / (mean_abs + eps)
    impulse_val = max_val  / (mean_abs + eps)

    fft_mag  = np.abs(np.fft.rfft(window))
    freqs    = np.fft.rfftfreq(n, d=1.0 / fs)
    freq_dom = freqs[np.argmax(fft_mag)]
    spec_energy  = np.sum(fft_mag ** 2)
    psd_norm     = fft_mag ** 2 / (spec_energy + eps)
    spec_entropy = -np.sum(psd_norm * np.log2(psd_norm + eps))

    return np.array([
        mean_val, std_val, rms_val, max_val, min_val,
        p2p_val, skew_val, kurt_val, crest_val, shape_val,
        impulse_val, freq_dom, spec_energy, spec_entropy,
    ], dtype=np.float32)


# ─── LECTURE DES CAPTEURS ───────────────────────────────────────────────────────

def reader_thread(port: str, baud: int, data_queue: queue.Queue,
                  stop_event: threading.Event) -> None:
    """Lit les samples depuis le port série -> data_queue (1 float par ligne,
    ou 1er champ si valeurs séparées par des virgules)."""
    try:
        ser = serial.Serial(port, baud, timeout=1.0)
        print(f"[Capteur] Connecté sur {port} @ {baud} baud")
        while not stop_event.is_set():
            line = ser.readline().decode("utf-8", errors="ignore").strip()
            if not line:
                continue
            try:
                value = float(line.split(",")[0])
                data_queue.put(value)
            except ValueError:
                pass
        ser.close()
    except serial.SerialException as e:
        print(f"[Capteur] Erreur série sur {port} : {e}")
        stop_event.set()


def simulate_reader(data_queue: queue.Queue, stop_event: threading.Event,
                    fs: int = SAMPLING_RATE) -> None:
    """Signal de vibration simulé (sain) : sinusoïde 120 Hz + bruit blanc."""
    print("[Simulation] Génération de signal synthétique (120 Hz + bruit)")
    t = 0
    dt = 1.0 / fs
    while not stop_event.is_set():
        value = 0.5 * np.sin(2 * np.pi * 120 * t) + 0.05 * np.random.randn()
        data_queue.put(float(value))
        t += dt
        if int(t * fs) % 100 == 0:
            time.sleep(100 / fs)


def _start_reader(cfg: dict):
    """Démarre le thread de lecture adapté (réel ou simulé). Retourne
    (data_queue, stop_event, thread)."""
    data_queue = queue.Queue(maxsize=100_000)
    stop_event = threading.Event()
    if cfg.get("simulate", False):
        t = threading.Thread(target=simulate_reader,
                             args=(data_queue, stop_event), daemon=True)
    else:
        t = threading.Thread(target=reader_thread,
                             args=(cfg.get("port"), cfg.get("baud", 115200),
                                   data_queue, stop_event), daemon=True)
    t.start()
    return data_queue, stop_event, t


# ─── 1) RODAGE GLOBAL (une seule fois, avant tous les capteurs) ─────────────────

def global_warmup(duration: int = WARMUP_DURATION) -> None:
    """Attend `duration` secondes, le temps que la machine se stabilise.
    Commun à tous les capteurs : ne se fait qu'une fois."""
    print("=" * 70)
    print(f"  RODAGE GLOBAL — {duration // 60} min (stabilisation machine)")
    print("=" * 70)
    start = time.time()
    while time.time() - start < duration:
        remaining = duration - (time.time() - start)
        print(f"\r  {int(remaining // 60):02d}:{int(remaining % 60):02d} restantes   ",
              end="", flush=True)
        time.sleep(min(5, max(0.2, duration / 10)))
    print("\n[Rodage] Terminé ✓\n")


# ─── 2a) COLLECTE D'UN CAPTEUR (30 min) ─────────────────────────────────────────

def collect_one_sensor(name: str, cfg: dict) -> dict | None:
    """Collecte + extraction des features pour UN capteur (le rodage est global
    et déjà fait). Sauvegarde features_<name>.npy et meta_<name>.json."""

    source = "SIMULATION" if cfg.get("simulate") else cfg.get("port")
    print("=" * 70)
    print(f"  COLLECTE : {name}   (source : {source})   — {COLLECT_DURATION // 60} min")
    print("=" * 70)

    data_queue, stop_event, t = _start_reader(cfg)
    buffer        = []
    features_list = []

    collect_start = time.time()
    while time.time() - collect_start < COLLECT_DURATION:
        while not data_queue.empty() and len(buffer) < WINDOW_SIZE * 4:
            buffer.append(data_queue.get_nowait())
        while len(buffer) >= WINDOW_SIZE:
            window = np.array(buffer[:WINDOW_SIZE], dtype=np.float32)
            buffer = buffer[WINDOW_STEP:]
            features_list.append(extract_features(window))
        remaining = COLLECT_DURATION - (time.time() - collect_start)
        print(f"\r  {int(remaining // 60):02d}:{int(remaining % 60):02d} restantes "
              f"| {len(features_list)} fenêtres   ", end="", flush=True)
        time.sleep(0.1)

    stop_event.set()
    t.join(timeout=2.0)
    print(f"\n[Collecte] Terminé ✓ — {len(features_list)} fenêtres extraites")

    if not features_list:
        print(f"[ERREUR] Aucune feature extraite pour '{name}'. Capteur ignoré.\n")
        return None

    features_array = np.stack(features_list, axis=0)
    feat_file = OUTPUT_DIR / f"features_{name}.npy"
    meta_file = OUTPUT_DIR / f"meta_{name}.json"
    np.save(feat_file, features_array)

    meta = {
        "sensor_name"     : name,
        "source"          : source,
        "n_windows"       : int(features_array.shape[0]),
        "n_features"      : int(features_array.shape[1]),
        "feature_names"   : FEATURE_NAMES,
        "window_size"     : WINDOW_SIZE,
        "window_step"     : WINDOW_STEP,
        "sampling_rate_hz": SAMPLING_RATE,
        "collected_at"    : time.strftime("%Y-%m-%dT%H:%M:%S"),
    }
    with open(meta_file, "w") as f:
        json.dump(meta, f, indent=2)
    print(f"[Sauvegarde] {feat_file}  shape={features_array.shape}  + {meta_file}\n")

    return {"name": name, "features": features_array}


# ─── 2b) ENTRAÎNEMENT D'UN CAPTEUR (autoencodeur) ───────────────────────────────

def train_one_sensor(name: str, k_sigma: float = K_SIGMA) -> dict | None:
    """Entraîne un autoencodeur sur features_<name>.npy, calcule le seuil
    d'anomalie, sauvegarde model_<name>.keras et threshold_<name>.json."""
    from sklearn.preprocessing import RobustScaler   # import à la demande
    from tensorflow import keras

    feat_file = OUTPUT_DIR / f"features_{name}.npy"
    if not feat_file.exists():
        print(f"[Entraînement] {feat_file} introuvable -> '{name}' ignoré.\n")
        return None

    print("=" * 70)
    print(f"  ENTRAÎNEMENT : {name}")
    print("=" * 70)

    features = np.load(feat_file)
    scaler   = RobustScaler()
    X        = scaler.fit_transform(features).astype(np.float32)
    n_feat   = X.shape[1]

    # Autoencodeur 14 -> 8 -> 4 -> 8 -> 14
    inputs  = keras.Input(shape=(n_feat,))
    encoded = keras.layers.Dense(8, activation="relu")(inputs)
    encoded = keras.layers.Dense(4, activation="relu")(encoded)   # bottleneck
    decoded = keras.layers.Dense(8, activation="relu")(encoded)
    outputs = keras.layers.Dense(n_feat, activation="linear")(decoded)
    ae = keras.Model(inputs, outputs, name=f"ae_{name}")
    ae.compile(optimizer="adam", loss="mse")

    ae.fit(
        X, X, epochs=EPOCHS, batch_size=BATCH_SIZE,
        validation_split=0.1, shuffle=True,
        callbacks=[keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=5, restore_best_weights=True)],
        verbose=0,
    )

    # Seuil d'anomalie sur l'erreur de reconstruction
    recon  = ae.predict(X, verbose=0)
    errors = np.mean((X - recon) ** 2, axis=1)
    mean_err = float(errors.mean())
    std_err  = float(errors.std())
    threshold = mean_err + k_sigma * std_err

    model_file = OUTPUT_DIR / f"model_{name}.keras"
    thr_file   = OUTPUT_DIR / f"threshold_{name}.json"
    ae.save(model_file)
    with open(thr_file, "w") as f:
        json.dump({
            "sensor_name"  : name,
            "mean_error"   : mean_err,
            "std_error"    : std_err,
            "k_sigma"      : k_sigma,
            "threshold"    : threshold,
            "scaler_center": scaler.center_.tolist(),
            "scaler_scale" : scaler.scale_.tolist(),
        }, f, indent=2)

    print(f"[Entraînement] {name} : seuil = {threshold:.6f} "
          f"(mean={mean_err:.6f}, std={std_err:.6f})")
    print(f"[Sauvegarde] {model_file}  + {thr_file}\n")
    return {"name": name, "threshold": threshold}


# ─── 3) DÉTECTION DE PROBLÈME (inférence) ───────────────────────────────────────

def detect_one_sensor(name: str, cfg: dict,
                      duration: int = DETECT_DURATION) -> dict | None:
    """Surveille un capteur pendant `duration` s, compare l'erreur de
    reconstruction au seuil, et rend un verdict."""
    from tensorflow import keras   # import à la demande

    model_file = OUTPUT_DIR / f"model_{name}.keras"
    thr_file   = OUTPUT_DIR / f"threshold_{name}.json"
    if not model_file.exists() or not thr_file.exists():
        print(f"[Détection] Modèle/seuil manquant pour '{name}' -> ignoré.\n")
        return None

    model = keras.models.load_model(model_file)
    with open(thr_file) as f:
        thr = json.load(f)
    center    = np.array(thr["scaler_center"], dtype=np.float32)
    scale     = np.array(thr["scaler_scale"],  dtype=np.float32)
    threshold = float(thr["threshold"])
    eps = 1e-10

    source = "SIMULATION" if cfg.get("simulate") else cfg.get("port")
    print("=" * 70)
    print(f"  DÉTECTION : {name}   (source : {source})   — {duration}s de surveillance")
    print("=" * 70)

    data_queue, stop_event, t = _start_reader(cfg)
    buffer = []
    errors = []

    start = time.time()
    while time.time() - start < duration:
        while not data_queue.empty() and len(buffer) < WINDOW_SIZE * 4:
            buffer.append(data_queue.get_nowait())
        while len(buffer) >= WINDOW_SIZE:
            window = np.array(buffer[:WINDOW_SIZE], dtype=np.float32)
            buffer = buffer[WINDOW_STEP:]
            feat        = extract_features(window).reshape(1, -1)
            feat_scaled = (feat - center) / (scale + eps)
            recon       = model(feat_scaled, training=False).numpy()  # rapide en streaming
            err         = float(np.mean((feat_scaled - recon) ** 2))
            errors.append(err)
            flag = "ANOMALIE !" if err > threshold else "ok"
            print(f"\r  fenêtres={len(errors)}  err={err:.4f}  seuil={threshold:.4f}  [{flag}]   ",
                  end="", flush=True)
        time.sleep(0.05)

    stop_event.set()
    t.join(timeout=2.0)

    errors = np.array(errors) if errors else np.array([0.0])
    n_anom = int(np.sum(errors > threshold))
    frac   = n_anom / len(errors)
    verdict = "PROBLÈME DÉTECTÉ" if frac > ANOMALY_FRACTION else "RAS (sain)"
    print(f"\n[Détection] {name} : {n_anom}/{len(errors)} fenêtres > seuil "
          f"({frac*100:.1f}%)  ->  {verdict}\n")

    return {"name": name, "n_windows": int(len(errors)),
            "n_anomalies": n_anom, "fraction": frac, "verdict": verdict}


# ─── ORCHESTRATION : PIPELINE COMPLÈTE EN SÉRIE ─────────────────────────────────

def run_pipeline(sensors: list, pause_between: bool = False) -> list:
    """1) Rodage global -> 2) par capteur en série : collecte puis entraînement
    -> 3) détection de problème en série."""
    n = len(sensors)

    # 1) Rodage global (une seule fois)
    global_warmup(WARMUP_DURATION)

    # 2) Collecte + entraînement, capteur par capteur
    for i, cfg in enumerate(sensors, start=1):
        name = cfg["name"]
        print("\n" + "#" * 70)
        print(f"#  CAPTEUR {i}/{n} : {name}   —   COLLECTE puis ENTRAÎNEMENT")
        print("#" * 70)
        if pause_between and i > 1 and not cfg.get("simulate", False):
            input(f"→ Prépare le capteur '{name}' puis appuie sur [Entrée]...")
        col = collect_one_sensor(name, cfg)
        if col is not None:
            train_one_sensor(name)

    # 3) Détection de problème, en série
    print("\n" + "#" * 70)
    print("#  PHASE DÉTECTION — capteurs analysés en série")
    print("#" * 70)
    detections = []
    for cfg in sensors:
        res = detect_one_sensor(cfg["name"], cfg)
        if res is not None:
            detections.append(res)

    print_detection_summary(detections)
    return detections


def print_detection_summary(detections: list) -> None:
    """Tableau récapitulatif des verdicts de détection."""
    if not detections:
        print("Aucun résultat de détection.")
        return
    print("\n" + "=" * 70)
    print("  RÉCAPITULATIF — détection de problème (en série)")
    print("=" * 70)
    header = f"{'Capteur':<14}{'Fenêtres':>10}{'Anomalies':>12}{'%':>8}   {'Verdict'}"
    print(header)
    print("-" * 70)
    for d in detections:
        print(f"{d['name']:<14}{d['n_windows']:>10}{d['n_anomalies']:>12}"
              f"{d['fraction']*100:>7.1f}%   {d['verdict']}")
    print("=" * 70 + "\n")


# ─── LANCEMENT ──────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # ▶ Test sans matériel (rodage + collecte + entraînement + détection, simulés) :
    run_pipeline(SENSORS_SIM, pause_between=False)

    # ▶ Avec les vrais capteurs (tous branchés, un port COM chacun) :
    # run_pipeline(SENSORS, pause_between=False)
